# SEC EDGAR Financial Data — Ingestion Pipeline
**Project:** SEC EDGAR Financial Ratio Analysis  

---

### Pipeline Overview
This notebook implements a production-grade ingestion pipeline for SEC EDGAR DERA 
Financial Statement Data Sets. It downloads, validates, and persists structured 
financial filing data as Delta tables for downstream analysis.

### Data Coverage
| Period | Range | Quarters |
|--------|-------|----------|
| Pre-COVID | Q1 2018 — Q4 2019 | 8 |
| COVID Impact | Q1 2020 — Q2 2020 | 2 |
| Post-COVID Recovery | Q3 2020 — Q1 2025 | 19 |
| **Total** | | **29** |

### Tables Produced
- `edgar_sub_{year}_Q{quarter}` → Filing metadata (company, SIC, dates)
- `edgar_num_{year}_Q{quarter}` → Financial numeric values (ratio inputs)

## Step 1 — Environment Setup
Install and validate all required dependencies.


In [0]:
%pip install yfinance requests --quiet

## Step 2 — Imports & Spark Initialization
Initialize Spark session and validate runtime environment.

In [0]:
import requests
import zipfile
import io
import os
import time
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType
import yfinance as yf

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("edgar_ingestion")

# Runtime metadata
PIPELINE_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ENGINEER = "Meet Saini"
SEC_USER_AGENT = "Meet Saini meetsaini3221@gmail.com"
BASE_URL = "https://www.sec.gov/files/dera/data/financial-statement-data-sets"
TARGET_FILES = ["sub.txt", "num.txt"]
RATE_LIMIT_SLEEP = 2  # seconds between requests (SEC policy)

print(f"✅ Spark session initialized | Version: {spark.version}")
print(f"✅ Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"✅ Target files: {TARGET_FILES}")

## Step 3 — Quarter Range Configuration
Define all 29 quarters with period classification labels.  
Period labels are used downstream for pre/post COVID comparative analysis.

In [0]:
def get_period_label(year: int, quarter: int) -> str:
    """Classifies quarter into COVID period bucket."""
    if year < 2020:
        return "Pre-COVID"
    elif year == 2020 and quarter <= 2:
        return "COVID-Impact"
    else:
        return "Post-COVID-Recovery"


def build_quarter_manifest() -> list:
    """
    Builds smart manifest — skips only fully ingested quarters.
    A quarter is complete only when BOTH sub AND num tables exist.
    Target years: 2018, 2019, 2020, 2021, 2022, 2023, 2024
    """
    all_quarters = []
    target_years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
    
    for year in target_years:
        for q in range(1, 5):
            all_quarters.append((year, q))
    
    # Get all existing tables as lowercase
    existing_tables = [
        row["tableName"].lower().strip()
        for row in spark.sql("SHOW TABLES LIKE 'edgar*'").collect()
    ]
    
    print("Existing tables found:")
    for t in sorted(existing_tables):
        print(f"  {t}")
    print()
    
    # Queue quarter only if BOTH sub AND num are missing
    remaining = []
    for year, q in all_quarters:
        sub_exists = f"edgar_sub_{year}_q{q}" in existing_tables
        num_exists = f"edgar_num_{year}_q{q}" in existing_tables

        if sub_exists and num_exists:
            print(f"⏭️  Skipping {year} Q{q} — both sub & num already ingested")
        elif sub_exists and not num_exists:
            print(f"⚠️  Partial {year} Q{q} — sub exists but num MISSING → queued")
            remaining.append((year, q))
        elif not sub_exists and num_exists:
            print(f"⚠️  Partial {year} Q{q} — num exists but sub MISSING → queued")
            remaining.append((year, q))
        else:
            print(f"📋 Queued  {year} Q{q} → {get_period_label(year, q)}")
            remaining.append((year, q))
    
    print(f"\n✅ Already done : {len(all_quarters) - len(remaining)}")
    print(f"📋 Remaining    : {len(remaining)}")
    return remaining


# Build manifest
quarters = build_quarter_manifest()

## Step 4 — Schema Validation Utility
Validates that downloaded files conform to expected DERA schema before persisting.  
Prevents corrupt or malformed data from entering the Delta tables.

In [0]:
# Expected minimum columns for each file
EXPECTED_SCHEMA = {
    "sub.txt": ["adsh", "cik", "name", "sic", "countryba", "period", "filed"],
    "num.txt": ["adsh", "tag", "version", "ddate", "qtrs", "uom", "value"]
}

def validate_schema(header: list, file_name: str) -> bool:
    """
    Validates that required columns exist in the downloaded file header.
    
    Args:
        header: List of column names from file
        file_name: Name of the file being validated
    
    Returns:
        True if valid, raises ValueError if not
    """
    expected = EXPECTED_SCHEMA.get(file_name, [])
    missing = [col for col in expected if col not in header]
    
    if missing:
        raise ValueError(
            f"Schema validation failed for {file_name}. "
            f"Missing columns: {missing}"
        )
    return True


def clean_rows(rows: list, header: list) -> list:
    """
    Filters out malformed rows that don't match header length.
    
    Args:
        rows: Raw parsed rows
        header: Expected header columns
    
    Returns:
        Cleaned list of valid rows
    """
    original_count = len(rows)
    cleaned = [r for r in rows if len(r) == len(header)]
    dropped = original_count - len(cleaned)
    
    if dropped > 0:
        logger.warning(f"Dropped {dropped} malformed rows ({dropped/original_count:.1%})")
    
    return cleaned

print("✅ Schema validation utilities defined")
print(f"   sub.txt expected columns: {EXPECTED_SCHEMA['sub.txt']}")
print(f"   num.txt expected columns: {EXPECTED_SCHEMA['num.txt']}")

## Step 5 — Core Ingestion Function
Downloads a single quarter zip from SEC EDGAR, validates schema,
adds metadata columns, and persists as Delta tables.

**Metadata columns added to every table:**
- `period` → COVID period classification
- `ingestion_timestamp` → Pipeline run timestamp
- `pipeline_run_id` → Unique run identifier for lineage tracking

In [0]:
def download_dera_quarter(year: int, quarter: int) -> dict:
    """
    Downloads, validates, and persists DERA financial data for a single quarter.
    Skips individual files that already exist as Delta tables.
    """
    url = f"{BASE_URL}/{year}q{quarter}.zip"
    period = get_period_label(year, quarter)
    results = {
        "year": year,
        "quarter": quarter,
        "period": period,
        "status": None,
        "tables": {}
    }
    
    headers = {
        "User-Agent": SEC_USER_AGENT,
        "Accept-Encoding": "gzip, deflate"
    }
    
    # Get existing tables
    existing_tables = [
        row["tableName"].lower().strip()
        for row in spark.sql("SHOW TABLES LIKE 'edgar*'").collect()
    ]
    
    # Determine which files still need downloading
    files_to_download = []
    for file_name in TARGET_FILES:
        table_name = f"edgar_{file_name.replace('.txt', '')}_{year}_q{quarter}"
        if table_name.lower() in existing_tables:
            logger.info(f"⏭️  Skipping {file_name} for {year} Q{quarter} — already exists")
        else:
            files_to_download.append(file_name)
    
    if not files_to_download:
        logger.info(f"⏭️  All files already ingested for {year} Q{quarter}")
        results["status"] = "SKIPPED"
        return results
    
    logger.info(f"Starting ingestion | {year} Q{quarter} | {period} | Files: {files_to_download}")
    
    try:
        response = requests.get(url, headers=headers, timeout=60)
        response.raise_for_status()
        
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            available_files = z.namelist()
            logger.info(f"ZIP contents: {available_files}")
            
            for file_name in files_to_download:
                if file_name not in available_files:
                    logger.warning(f"File not found in ZIP: {file_name} — skipping")
                    continue
                
                content = z.read(file_name).decode("utf-8", errors="replace")
                rows = [line.split("\t") for line in content.split("\n") if line.strip()]
                
                if len(rows) < 2:
                    logger.warning(f"Empty file: {file_name} — skipping")
                    continue
                
                header = rows[0]
                data = rows[1:]
                
                # Validate schema
                validate_schema(header, file_name)
                
                # Clean rows
                data = clean_rows(data, header)
                
                # Create DataFrame
                df = spark.createDataFrame(data, schema=header)
                
                # Add metadata columns
                df = (df
                    .withColumn("period", lit(period))
                    .withColumn("source_year", lit(year))
                    .withColumn("source_quarter", lit(quarter))
                    .withColumn("ingestion_timestamp", current_timestamp())
                    .withColumn("pipeline_run_id", lit(PIPELINE_RUN_ID))
                )
                
                # Persist as Delta table
                table_name = f"edgar_{file_name.replace('.txt', '')}_{year}_Q{quarter}"
                df.write.format("delta").mode("overwrite").saveAsTable(table_name)
                
                row_count = df.count()
                results["tables"][table_name] = row_count
                logger.info(f"✅ Saved: {table_name} | Rows: {row_count:,}")
        
        results["status"] = "SUCCESS"
        time.sleep(RATE_LIMIT_SLEEP)
        
    except requests.exceptions.HTTPError as e:
        results["status"] = f"HTTP_ERROR: {e}"
        logger.error(f"❌ HTTP error for {year} Q{quarter}: {e}")
    except zipfile.BadZipFile as e:
        results["status"] = f"ZIP_ERROR: {e}"
        logger.error(f"❌ ZIP error for {year} Q{quarter}: {e}")
    except ValueError as e:
        results["status"] = f"SCHEMA_ERROR: {e}"
        logger.error(f"❌ Schema error for {year} Q{quarter}: {e}")
    except Exception as e:
        results["status"] = f"UNKNOWN_ERROR: {e}"
        logger.error(f"❌ Unexpected error for {year} Q{quarter}: {e}")
    
    return results

print("✅ Core ingestion function defined — smart file-level skip enabled")

## Step 6 — Single Quarter Test
Validate the pipeline end-to-end with a single quarter before running the full loop.  
Always test before bulk operations.

In [0]:
# Smoke test — 2020 Q1 (COVID Impact period)
#test_result = download_dera_quarter(2020, 1)

#print("\n--- Test Result ---")
#print(f"Status  : {test_result['status']}")
#print(f"Period  : {test_result['period']}")
#print(f"Tables  :")
#for table, rows in test_result["tables"].items():
 #   print(f"  📋 {table} → {rows:,} rows")

In [0]:
audit_log = []
failed_quarters = []

print(f"{'='*50}")
print(f"EDGAR INGESTION PIPELINE — RUN ID: {PIPELINE_RUN_ID}")
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total Quarters: {len(quarters)}")
print(f"{'='*50}\n")

for i, (year, quarter) in enumerate(quarters, 1):
    print(f"[{i:02d}/{len(quarters)}] Processing {year} Q{quarter}...")
    result = download_dera_quarter(year, quarter)
    audit_log.append(result)
    
    if result["status"] != "SUCCESS":
        failed_quarters.append((year, quarter))

# Retry failed quarters once
if failed_quarters:
    print(f"\n⚠️ Retrying {len(failed_quarters)} failed quarters...")
    time.sleep(10)
    for year, quarter in failed_quarters.copy():
        result = download_dera_quarter(year, quarter)
        if result["status"] == "SUCCESS":
            failed_quarters.remove((year, quarter))
            print(f"  ✅ Retry succeeded: {year} Q{quarter}")

# Final summary
print(f"\n{'='*50}")
print("INGESTION SUMMARY")
print(f"{'='*50}")
print(f"Run ID      : {PIPELINE_RUN_ID}")
print(f"End Time    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total       : {len(quarters)} quarters")
print(f"✅ Success  : {len(quarters) - len(failed_quarters)}")
print(f"❌ Failed   : {len(failed_quarters)}")
if failed_quarters:
    print(f"Failed list : {failed_quarters}")
print(f"{'='*50}")